# SPX Implied Volatility Surface Reconstruction

**Author:** Jonathan Phung (ESCP Business School, MSc Market Finance)

This notebook rebuilds an implied volatility surface from a snapshot of listed SPX option quotes, instead of using a precomputed volatility dataset.

**Pipeline**
1. Load the option chain and market metadata
2. Clean quotes (crossed / zero bids, wide spreads, duplicates, very short maturities)
3. Match calls and puts by strike and expiry
4. Estimate one forward per expiry from put-call parity
5. Keep out-of-the-money options (puts below the forward, calls above)
6. Invert Black-76 to get implied volatilities
7. Sanity checks: calendar arbitrage (total variance) and butterfly arbitrage (call price convexity)
8. Visualise smiles, the 3D surface and a heatmap

**Inputs**
- `Chain.csv`: one row per option quote with columns `expiry, strike, type (C/P), bid, ask` (optional: `volume, open_interest`)
- `Meta.csv`: one row with `valuation_date, spot, rate` (continuously compounded, flat)

If the files are not found, the notebook generates a synthetic chain so that every cell runs end to end. **Replace it with real SPX quotes before drawing any conclusion.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import brentq
from scipy.stats import norm

pd.set_option("display.width", 120)
DATA_DIR = Path(".")

## 1. Load data

In [ ]:
def make_synthetic_chain(seed=0):
    # Synthetic SPX-like chain priced from a skewed smile. Used ONLY if Chain.csv / Meta.csv are missing.
    rng = np.random.default_rng(seed)
    spot, rate, q = 5000.0, 0.045, 0.013
    val_date = pd.Timestamp("2026-01-02")
    expiries = val_date + pd.to_timedelta([14, 30, 60, 91, 182, 273, 365], unit="D")
    rows = []
    for exp in expiries:
        T = (exp - val_date).days / 365.0
        F = spot * np.exp((rate - q) * T)
        df = np.exp(-rate * T)
        for K in np.arange(0.70, 1.30, 0.0125) * spot:
            k = np.log(K / F)
            vol = 0.17 - 0.35 * k / np.sqrt(max(T, 0.08)) * 0.25 + 0.6 * k**2 + 0.01 * np.sqrt(T)
            vol = max(vol, 0.06)
            for typ in ("C", "P"):
                px = black76_price(F, K, T, vol, df, typ)
                half_spread = max(0.05, 0.02 * px) * (1 + rng.uniform(0, 0.5))
                mid = px * (1 + rng.normal(0, 0.005))
                rows.append([exp.date(), round(K, 2), typ, max(mid - half_spread, 0), mid + half_spread])
    chain = pd.DataFrame(rows, columns=["expiry", "strike", "type", "bid", "ask"])
    meta = pd.DataFrame([{"valuation_date": val_date.date(), "spot": spot, "rate": rate}])
    return chain, meta


def black76_price(F, K, T, sigma, df, option_type):
    # Black-76 price of a European option on a forward F, discounted with df.
    if T <= 0 or sigma <= 0:
        intrinsic = max(F - K, 0) if option_type == "C" else max(K - F, 0)
        return df * intrinsic
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "C":
        return df * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return df * (K * norm.cdf(-d2) - F * norm.cdf(-d1))


chain_path, meta_path = DATA_DIR / "Chain.csv", DATA_DIR / "Meta.csv"
if chain_path.exists() and meta_path.exists():
    chain = pd.read_csv(chain_path)
    meta = pd.read_csv(meta_path)
    SYNTHETIC = False
else:
    chain, meta = make_synthetic_chain()
    SYNTHETIC = True
    print("Chain.csv / Meta.csv not found: using a SYNTHETIC chain for demonstration.")

spot = float(meta["spot"].iloc[0])
rate = float(meta["rate"].iloc[0])
val_date = pd.Timestamp(meta["valuation_date"].iloc[0])

chain["expiry"] = pd.to_datetime(chain["expiry"])
chain["type"] = chain["type"].str.upper().str[0]
chain["T"] = (chain["expiry"] - val_date).dt.days / 365.0
print(f"Spot {spot:,.2f} | rate {rate:.2%} | valuation {val_date.date()} | raw quotes {len(chain):,}")
chain.head()

## 2. Clean the quotes

Rules (deliberately simple and explicit):
- drop zero or crossed bids/asks and non-positive maturities
- drop maturities under 7 days (pin risk, very noisy implied vols)
- drop quotes whose relative spread exceeds 50% of mid
- keep a single quote per (expiry, strike, type)

In [ ]:
MIN_DAYS, MAX_REL_SPREAD = 7, 0.50

c = chain.copy()
c["mid"] = (c["bid"] + c["ask"]) / 2
c["rel_spread"] = (c["ask"] - c["bid"]) / c["mid"]

mask = (
    (c["bid"] > 0) & (c["ask"] > c["bid"]) & (c["T"] * 365 >= MIN_DAYS)
    & (c["rel_spread"] <= MAX_REL_SPREAD)
)
clean = (c[mask]
         .sort_values("rel_spread")
         .drop_duplicates(["expiry", "strike", "type"])
         .sort_values(["expiry", "strike", "type"])
         .reset_index(drop=True))
print(f"Raw quotes: {len(chain):,} -> clean quotes: {len(clean):,}")

## 3. Match calls and puts, estimate the forward

For a European call C and put P with the same strike K and maturity T:

$$C - P = e^{-rT}(F - K) \quad\Rightarrow\quad F = K + e^{rT}(C - P)$$

One forward per expiry is taken as the median of this estimate over the strikes closest to spot, where both quotes are the most liquid. The implied dividend yield falls out of it.

In [ ]:
calls = clean[clean["type"] == "C"].set_index(["expiry", "strike"])
puts = clean[clean["type"] == "P"].set_index(["expiry", "strike"])
pairs = calls[["T", "mid"]].join(puts[["mid"]], lsuffix="_call", rsuffix="_put", how="inner").reset_index()
pairs["forward_est"] = pairs["strike"] + np.exp(rate * pairs["T"]) * (pairs["mid_call"] - pairs["mid_put"])

N_ATM = 6  # strikes closest to spot used per expiry
fwd = (pairs.assign(dist=(pairs["strike"] - spot).abs())
            .sort_values("dist")
            .groupby("expiry")
            .head(N_ATM)
            .groupby("expiry")
            .agg(T=("T", "first"), forward=("forward_est", "median"), n_pairs=("forward_est", "size")))
fwd["implied_div_yield"] = rate - np.log(fwd["forward"] / spot) / fwd["T"]
print(f"Matched call/put pairs: {len(pairs):,}")
fwd.round(4)

## 4. Keep out-of-the-money options

OTM options carry most of the time value and are the most liquid on each side of the forward: puts for K < F, calls for K >= F. This also avoids deep in-the-money quotes whose implied vols are very sensitive to small price errors.

In [ ]:
clean = clean.merge(fwd[["forward"]], left_on="expiry", right_index=True, how="inner")
clean["log_moneyness"] = np.log(clean["strike"] / clean["forward"])
clean["is_otm"] = np.where(clean["type"] == "P", clean["strike"] < clean["forward"], clean["strike"] >= clean["forward"])
otm = clean[clean["is_otm"]].copy()
print(f"OTM quotes kept: {len(otm):,}")

## 5. Invert Black-76

For each OTM quote, solve $\text{Black76}(F, K, T, \sigma) = \text{mid}$ for $\sigma$ with Brent's method on $[10^{-4}, 5]$. Quotes below the discounted intrinsic value or above the no-arbitrage upper bound have no solution and are dropped.

In [ ]:
def implied_vol(price, F, K, T, df, option_type):
    intrinsic = df * (max(F - K, 0) if option_type == "C" else max(K - F, 0))
    upper = df * (F if option_type == "C" else K)
    if not (intrinsic < price < upper):
        return np.nan
    f = lambda s: black76_price(F, K, T, s, df, option_type) - price
    try:
        return brentq(f, 1e-4, 5.0, xtol=1e-8)
    except ValueError:
        return np.nan

otm["df"] = np.exp(-rate * otm["T"])
otm["iv"] = [implied_vol(r.mid, r.forward, r.strike, r.T, r.df, r.type) for r in otm.itertuples()]
surface = otm.dropna(subset=["iv"]).copy()
surface["total_var"] = surface["iv"] ** 2 * surface["T"]
print(f"Implied vols solved: {len(surface):,} / {len(otm):,}")
surface[["expiry", "strike", "type", "mid", "forward", "log_moneyness", "iv"]].head()

## 6. Sanity checks

**Calendar arbitrage:** at a fixed log-moneyness, total implied variance $w = \sigma^2 T$ must not decrease with maturity.

**Butterfly arbitrage:** for a given expiry, call prices must be convex in strike. OTM puts are converted to calls with put-call parity before testing.

These are diagnostics only: violations are reported, not repaired.

In [ ]:
# Calendar check on a common log-moneyness grid
grid_k = np.linspace(-0.15, 0.10, 11)
w = {}
for exp, g in surface.groupby("expiry"):
    g = g.sort_values("log_moneyness")
    if g["log_moneyness"].min() <= grid_k[0] and g["log_moneyness"].max() >= grid_k[-1]:
        w[exp] = np.interp(grid_k, g["log_moneyness"], g["total_var"])
w = pd.DataFrame(w, index=np.round(grid_k, 3)).T.sort_index()
calendar_violations = int((w.diff().iloc[1:] < -1e-6).sum().sum())

# Butterfly check: convexity of call prices in strike
butterfly_violations = 0
for exp, g in surface.groupby("expiry"):
    g = g.sort_values("strike")
    call_px = np.where(g["type"] == "C", g["mid"], g["mid"] + g["df"] * (g["forward"] - g["strike"]))
    K = g["strike"].to_numpy()
    if len(K) < 3:
        continue
    slopes = np.diff(call_px) / np.diff(K)
    butterfly_violations += int((np.diff(slopes) < -1e-6).sum())

print(f"Calendar violations (grid points): {calendar_violations}")
print(f"Butterfly violations (strike triplets): {butterfly_violations}")

## 7. Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for exp, g in surface.groupby("expiry"):
    g = g.sort_values("log_moneyness")
    ax.plot(g["log_moneyness"], g["iv"] * 100, marker=".", lw=1, label=f"{exp.date()} ({g['T'].iloc[0]*365:.0f}d)")
ax.axvline(0, color="grey", lw=0.8, ls="--")
ax.set_xlabel("log-moneyness  ln(K / F)")
ax.set_ylabel("implied volatility (%)")
ax.set_title("SPX implied volatility smiles by expiry" + (" (synthetic data)" if SYNTHETIC else ""))
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("spx_implied_volatility_smiles.png", dpi=150)
plt.show()

In [ ]:
from scipy.interpolate import griddata

k_grid = np.linspace(surface["log_moneyness"].quantile(0.02), surface["log_moneyness"].quantile(0.98), 60)
t_grid = np.linspace(surface["T"].min(), surface["T"].max(), 40)
KK, TT = np.meshgrid(k_grid, t_grid)
IV = griddata((surface["log_moneyness"], surface["T"]), surface["iv"], (KK, TT), method="linear")

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(KK, TT, IV * 100, cmap="viridis", edgecolor="none", alpha=0.9)
ax.scatter(surface["log_moneyness"], surface["T"], surface["iv"] * 100, s=3, color="black", alpha=0.4)
ax.set_xlabel("ln(K / F)")
ax.set_ylabel("maturity (years)")
ax.set_zlabel("implied vol (%)")
ax.set_title("SPX implied volatility surface (observed points + linear interpolation)")
plt.tight_layout()
plt.savefig("spx_implied_volatility_surface_3d.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(KK, TT, IV * 100, shading="auto", cmap="viridis")
ax.scatter(surface["log_moneyness"], surface["T"], s=4, color="white", alpha=0.5)
fig.colorbar(im, ax=ax, label="implied vol (%)")
ax.set_xlabel("ln(K / F)")
ax.set_ylabel("maturity (years)")
ax.set_title("Implied volatility heatmap (white dots = observed quotes)")
plt.tight_layout()
plt.savefig("spx_iv_surface_scatter_heatmap.png", dpi=150)
plt.show()

## 8. Summary and limitations

**Typical shape:** strong negative skew (OTM puts trade at higher implied vols than OTM calls) that flattens with maturity, the classic equity index pattern explained by crash protection demand and the negative spot/vol correlation.

**Limitations**
- single snapshot, mid prices, flat interest rate
- dividends only through the implied forward (no explicit dividend curve)
- linear interpolation between observed points, no parametric fit (SVI / SSVI would be the natural next step)
- arbitrage is diagnosed, not repaired

**Next steps:** fit SVI per expiry, enforce no-arbitrage (SSVI), compute local volatility (Dupire) from the fitted surface.